Chosen Track — B: Tabular Record Batch Scoring

This track uses three records from cleaned_data.csv and asks the LLM to return a validated JSON risk assessment for each record.

Task 1 — Install and Import Libraries

In [ ]:
!pip install -q requests jsonschema

In [ ]:
import os
import re
import json
import getpass
from pathlib import Path
from google.colab import userdata
import pandas as pd
import requests

from jsonschema import validate
from jsonschema.exceptions import ValidationError

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

Path("results").mkdir(exist_ok=True)

print("Libraries imported successfully.")

**Task 2 — Upload and Load cleaned_data.csv**

In [ ]:
df = pd.read_csv("cleaned_data.csv")

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

**Task 3 — Store the API Key Safely**

In [ ]:

LLM_URL = userdata.get("url")
LLM_MODEL = userdata.get("model")



**Task 4 — Create the Reusable call_llm() Function**

In [ ]:
def call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):
    """
    Call the OpenRouter chat-completions API.

    Returns:
        str: Assistant response content when successful.
        None: When the API call fails.
    """

    api_key = os.environ.get("LLM_API_KEY")

    if not api_key:
        print("Error: LLM_API_KEY is not available.")
        return None

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(
            LLM_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

    except requests.RequestException as error:
        print("Request error:", error)
        return None

    if response.status_code != 200:
        print("API request failed.")
        print("Status code:", response.status_code)
        print("Response:", response.text[:10000])
        return None

    try:
        response_data = response.json()

        return response_data[
            "choices"
        ][0]["message"]["content"]

    except (
        ValueError,
        KeyError,
        IndexError,
        TypeError
    ) as error:
        print("Could not parse the API response:", error)
        print("Raw response:", response.text[:1000])
        return None

**Task 5 — PII Guardrail**

In [ ]:

def has_pii(text):
    email_pattern = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
    phone_pattern = r"\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"

    return bool(
        re.search(email_pattern, text)
        or re.search(phone_pattern, text)
    )


def safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=512
):
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        max_tokens=max_tokens
    )

**Task 6 — Test the LLM Connection**

In [ ]:
test_system_prompt = (
    "Follow the user's instruction exactly."
)

test_user_prompt = (
    "Reply with only the word: hello"
)

test_response = safe_call_llm(
    system_prompt=test_system_prompt,
    user_prompt=test_user_prompt,
    temperature=0.0,
    max_tokens=20
)

print("Test response:", test_response)

Task 7 — Define the JSON Schema

In [ ]:
assessment_schema = {
    "type": "object",

    "properties": {
        "risk_tier": {
            "type": "string",
            "enum": [
                "low",
                "medium",
                "high"
            ]
        },

        "flag_for_review": {
            "type": "boolean"
        },

        "primary_signal": {
            "type": "string"
        },

        "confidence": {
            "type": "string",
            "enum": [
                "low",
                "medium",
                "high"
            ]
        },

        "recommended_action": {
            "type": "string"
        }
    },

    "required": [
        "risk_tier",
        "flag_for_review",
        "primary_signal",
        "confidence",
        "recommended_action"
    ],

    "additionalProperties": False
}
fallback_assessment = {
    "risk_tier": None,
    "flag_for_review": None,
    "primary_signal": None,
    "confidence": None,
    "recommended_action": None
}

Task 8 — Write the Few-Shot System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a structured health-record screening assistant for an
educational data-science demonstration.

Your task is to score a de-identified patient record using this rubric:

RISK-TIER RUBRIC

HIGH:
- heart_attack_risk is 1, or
- several concerning clinical signals appear together, such as
  high resting blood pressure, high cholesterol, exercise angina,
  diabetes, multiple major vessels, high BMI, or high oldpeak.

MEDIUM:
- there are some concerning signals, but the overall evidence is
  mixed or incomplete.

LOW:
- heart_attack_risk is 0 and the record contains few major warning
  signals.

FLAG-FOR-REVIEW RUBRIC

Set flag_for_review to true when:
- risk_tier is high, or
- diabetes, exercise angina, elevated oldpeak, multiple major vessels,
  or another concerning combination suggests professional review.

CONFIDENCE RUBRIC

high:
- several fields consistently support the assigned tier.

medium:
- the evidence supports the tier but is mixed.

low:
- important fields are missing or conflicting.

RECOMMENDED ACTION RULES

Use a short non-diagnostic action such as:
- "Routine monitoring"
- "Review risk factors"
- "Seek professional medical assessment"

Do not diagnose a medical condition.
Do not claim that the assessment replaces a clinician.

Return only one valid JSON object.
Do not use Markdown.
Do not use a code fence.
Do not add fields outside the required schema.

Required JSON fields:
{
  "risk_tier": "low|medium|high",
  "flag_for_review": true|false,
  "primary_signal": "string",
  "confidence": "low|medium|high",
  "recommended_action": "string"
}

WORKED EXAMPLE

Input:
{
  "age": 68,
  "resting_bp": 158,
  "cholesterol": 272,
  "exercise_angina": 1,
  "oldpeak": 3.2,
  "num_major_vessels": 2,
  "bmi": 31.0,
  "diabetes": 1,
  "heart_attack_risk": 1
}

Output:
{
  "risk_tier": "high",
  "flag_for_review": true,
  "primary_signal": "Multiple elevated cardiovascular risk indicators",
  "confidence": "high",
  "recommended_action": "Seek professional medical assessment"
}
""".strip()

Task 9 — User Prompt Template

In [ ]:
USER_PROMPT_TEMPLATE = """
Assess the following de-identified dataset record using the rubric
defined in the system prompt.

Return only the required JSON object.

Input record:
{record_json}
""".strip()
def create_user_prompt(record):
    record_json = json.dumps(
        record,
        indent=2,
        ensure_ascii=False
    )

    return USER_PROMPT_TEMPLATE.format(
        record_json=record_json
    )

Task 10 — Select Three Dataset Records

In [ ]:
low_risk_rows = df[
    df["heart_attack_risk"] == 0
]

high_risk_rows = df[
    df["heart_attack_risk"] == 1
]

if len(low_risk_rows) < 1 or len(high_risk_rows) < 2:
    raise ValueError(
        "The dataset does not contain enough records "
        "from both target classes."
    )
    selected_df = pd.concat(
    [
        low_risk_rows.iloc[[0]],
        high_risk_rows.iloc[[0]],
        high_risk_rows.iloc[[1]]
    ],
    ignore_index=True
)

display(selected_df)
columns_to_send = [
    "age",
    "gender",
    "chest_pain_type",
    "resting_bp",
    "cholesterol",
    "fasting_blood_sugar",
    "resting_ecg",
    "max_heart_rate",
    "exercise_angina",
    "oldpeak",
    "st_slope",
    "num_major_vessels",
    "thalassemia",
    "bmi",
    "smoking_status",
    "alcohol_consumption",
    "physical_activity",
    "family_history",
    "diabetes",
    "stress_level",
    "heart_attack_risk"
]

columns_to_send = [
    col for col in columns_to_send
    if col in selected_df.columns
]
def dataframe_row_to_dict(row):
    record = {}

    for column, value in row.items():

        if pd.isna(value):
            record[column] = None

        elif hasattr(value, "item"):
            record[column] = value.item()

        else:
            record[column] = value

    return record
    records = []

for _, row in selected_df[
    columns_to_send
].iterrows():

    records.append(
        dataframe_row_to_dict(row)
    )

print(json.dumps(records, indent=2))

Task 11 — Parse and Validate LLM Output

In [ ]:
def parse_and_validate_assessment(raw_response):
    """
    Parse JSON and validate it against assessment_schema.

    Returns:
        tuple:
        (
            validated_or_fallback_dict,
            validation_status,
            error_message
        )
    """

    if raw_response is None:
        return (
            fallback_assessment.copy(),
            "fail",
            "No response returned"
        )

    cleaned_response = raw_response.strip()

    try:
        parsed_response = json.loads(
            cleaned_response
        )

    except json.JSONDecodeError as error:
        print("JSON parsing error:", error)

        return (
            fallback_assessment.copy(),
            "fail",
            f"JSONDecodeError: {error}"
        )

    try:
        validate(
            instance=parsed_response,
            schema=assessment_schema
        )

    except ValidationError as error:
        print(
            "Schema validation error:",
            error.message
        )

        return (
            fallback_assessment.copy(),
            "fail",
            f"ValidationError: {error.message}"
        )

    return (
        parsed_response,
        "pass",
        ""
    )

Task 12 — Run the End-to-End Pipeline on Three Records

In [ ]:
batch_results = []

for index, record in enumerate(
    records,
    start=1
):
    print("=" * 70)
    print(f"RECORD {index}")

    user_prompt = create_user_prompt(
        record
    )

    guardrail_status = (
        "blocked"
        if has_pii(user_prompt)
        else "passed"
    )

    raw_response = safe_call_llm(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=300
    )

    (
        assessment,
        validation_status,
        validation_error
    ) = parse_and_validate_assessment(
        raw_response
    )

    print("\nInput:")
    print(
        json.dumps(
            record,
            indent=2
        )
    )

    print("\nRaw LLM response:")
    print(raw_response)

    print("\nValidated assessment:")
    print(
        json.dumps(
            assessment,
            indent=2
        )
    )

    print(
        "\nValidation status:",
        validation_status
    )

    if validation_error:
        print(
            "Validation error:",
            validation_error
        )

    batch_results.append({
        "Input Number": index,
        "Input Record": json.dumps(
            record,
            ensure_ascii=False
        ),
        "LLM Assessment JSON": json.dumps(
            assessment,
            ensure_ascii=False
        ),
        "Validation Status":
            validation_status,
        "Validation Error":
            validation_error,
        "Guardrail Result":
            guardrail_status
    })
    batch_results_df = pd.DataFrame(
    batch_results
)

display(batch_results_df)
batch_results_df.to_csv(
    "results/batch_scoring_results.csv",
    index=False
)

Task 13 — Demonstrate the PII Guardrail

In [ ]:
pii_test_input = """
Assess this record.

Contact email: patient@example.com
Age: 59
Resting blood pressure: 140
"""

pii_test_response = safe_call_llm(
    system_prompt=SYSTEM_PROMPT,
    user_prompt=pii_test_input,
    temperature=0.0,
    max_tokens=100
)

print("PII test response:", pii_test_response)
clean_test_input = """
Return only this valid JSON object:
{
  "risk_tier": "low",
  "flag_for_review": false,
  "primary_signal": "No major warning signal in test input",
  "confidence": "low",
  "recommended_action": "Routine monitoring"
}
"""

clean_test_response = safe_call_llm(
    system_prompt=(
        "Return only valid JSON and follow "
        "the user's instruction."
    ),
    user_prompt=clean_test_input,
    temperature=0.0,
    max_tokens=150
)

print("Clean test response:")
print(clean_test_response)
guardrail_results = pd.DataFrame([
    {
        "Test": "Email-containing input",
        "Expected Result": "Blocked",
        "Actual Result":
            "Blocked"
            if pii_test_response is None
            else "Allowed"
    },
    {
        "Test": "Clean input",
        "Expected Result": "Allowed",
        "Actual Result":
            "Allowed"
            if clean_test_response is not None
            else "Failed or blocked"
    }
])

display(guardrail_results)

guardrail_results.to_csv(
    "results/guardrail_results.csv",
    index=False
)

Task 14 — Temperature A/B Comparison

In [ ]:
temperature_results = []

for index, record in enumerate(
    records,
    start=1
):
    user_prompt = create_user_prompt(
        record
    )

    output_temp_0 = safe_call_llm(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=300
    )

    output_temp_07 = safe_call_llm(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=0.7,
        max_tokens=300
    )

    key_difference = (
        "No textual difference"
        if output_temp_0 == output_temp_07
        else
        "Wording or assessment varied"
    )

    temperature_results.append({
        "Input": f"Record {index}",
        "Output at temp=0":
            output_temp_0,
        "Output at temp=0.7":
            output_temp_07,
        "Key difference":
            key_difference
    })
    temperature_comparison_df = pd.DataFrame(
    temperature_results
)

display(temperature_comparison_df)
temperature_comparison_df.to_csv(
    "results/temperature_comparison.csv",
    index=False
)

Task 15 — Validate Both Temperature Outputs

In [ ]:
temperature_validation_results = []

for row in temperature_results:

    for temperature_label in [
        "Output at temp=0",
        "Output at temp=0.7"
    ]:

        (
            parsed_output,
            status,
            error
        ) = parse_and_validate_assessment(
            row[temperature_label]
        )

        temperature_validation_results.append({
            "Input": row["Input"],
            "Temperature Output":
                temperature_label,
            "Validation Status": status,
            "Validation Error": error,
            "Parsed Output": json.dumps(
                parsed_output,
                ensure_ascii=False
            )
        })

temperature_validation_df = pd.DataFrame(
    temperature_validation_results
)

display(temperature_validation_df)
temperature_validation_df.to_csv(
    "results/temperature_validation_results.csv",
    index=False
)

Task 16 — Final Demonstration Table

In [ ]:
demonstration_table = batch_results_df[[
    "Input Number",
    "Input Record",
    "LLM Assessment JSON",
    "Validation Status",
    "Guardrail Result"
]].copy()

demonstration_table.columns = [
    "Input",
    "LLM Input Record",
    "LLM Output",
    "Valid JSON",
    "Pass/Block"
]

display(demonstration_table)
demonstration_table.to_csv(
    "results/final_demonstration_table.csv",
    index=False
)

Task 17 — Create requirements.txt

In [ ]:
requirements = """pandas
requests
jsonschema
"""

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(requirements)

print("requirements.txt created successfully.")
files.download("requirements.txt")

Download Result Files

In [ ]:
import shutil

shutil.make_archive(
    "part4_results",
    "zip",
    "results"
)

files.download("part4_results.zip")